In [1]:
import pandas as pd
import json

general_data = pd.read_csv("general_data_new.tsv", delimiter="\t")

# read work_id_to_journal_id.parquet


In [2]:
work_id_to_journal_id = pd.read_parquet("work_id_to_journal_id.parquet")

In [3]:
# Select only the specified columns
general_data = general_data[['id', 'cited_by_count', 'abstract_inverted_index']]

# Keep only distinct rows
general_data = general_data.drop_duplicates()

In [4]:
# Perform left join
general_data = pd.merge(general_data, work_id_to_journal_id, how='left', left_on='id', right_on='work_id')


In [57]:
# top_50_journals = pd.read_parquet("top_50_journals.parquet")
# filter general_data's journal_name to only include journals in top_50_journals (use display_name)
# general_data = general_data[general_data["journal_name"].isin(top_50_journals["display_name"])]
# general_data = general_data.sort_values(by='cited_by_count', ascending=False)

In [5]:
general_data

,id,cited_by_count,abstract_inverted_index,work_id,primary_location_id,journal_name
0,https://openalex.org/W1491339034,13,"{""(Depletion"": [102], ""(MCW)\u00ab\u00a0less"":...",https://openalex.org/W1491339034,https://openalex.org/S23254222,American Economic Review
1,https://openalex.org/W1481328804,2,"{""(1984),"": [4], ""(D*/P)"": [107], ""(assets),"":...",https://openalex.org/W1481328804,https://openalex.org/S23254222,American Economic Review
2,https://openalex.org/W2620990083,0,"{""But"": [36], ""I"": [0], ""I've"": [39], ""Or"": [2...",https://openalex.org/W2620990083,https://openalex.org/S142990027,Journal of Marketing
3,https://openalex.org/W2991892933,0,"{""1811"": [37], ""1974"": [68], ""However,"": [66],...",https://openalex.org/W2991892933,https://openalex.org/S108845946,Economica
4,https://openalex.org/W2392466246,0,"{""171"": [42], ""China"": [1, 15], ""China.Accordi...",https://openalex.org/W2392466246,https://openalex.org/S38825547,Economic Geography
...,...,...,...,...,...,...
389230,https://openalex.org/W3137007398,1,"{""Abstract"": [0], ""In"": [3], ""Pareto"": [47], ""...",https://openalex.org/W3137007398,https://openalex.org/S36178057,Economics Letters
389231,https://openalex.org/W4401487473,0,"{""(1992),"": [47], ""(at"": [85], ""G\u00fcth"": [4...",https://openalex.org/W4401487473,https://openalex.org/S94044085,Games and Economic Behavior
389232,https://openalex.org/W2088490799,75,"{""Although"": [0], ""DC,"": [54], ""El"": [59], ""I""...",https://openalex.org/W2088490799,https://openalex.org/S101209419,Journal of Development Economics
389233,https://openalex.org/W3098438983,74,"{""'green'"": [175], ""(AFNs)"": [134], ""(SFSCs).""...",https://openalex.org/W3098438983,https://openalex.org/S18051569,Resources Conservation and Recycling


In [6]:
# show counts in general_data by journal
general_data["journal_name"].value_counts()

journal_name
Journal of Political Economy                   14993
Economics Letters                              13664
The Journal of Economic History                13649
American Journal of Agricultural Economics     13183
The Economic Journal                           12764
                                               ...  
Public Choice                                    884
Family Business Review                           773
Brookings Papers on Economic Activity            641
Journal of International Business Studies        625
Journal of the Academy of Marketing Science      409
Name: count, Length: 100, dtype: int64

In [7]:
# Read the title_abstract.json file
import json
with open("title_abstract_new.json", "r", encoding="utf-8") as file:
    title_abstract_data = json.load(file)


In [8]:
# print object type of title_abstract_data
print(type(title_abstract_data))


<class 'list'>


In [9]:
len(title_abstract_data), title_abstract_data[:5]

(1018015,
 [{'id': 'https://openalex.org/W1491339034',
   'title': 'Public policies toward the use of scrap materials',
   'abstract': 'Proposals that have been considered to stimulate the flow of recycled materials are discussed. The thrust of proposals is that recycling rates are too low and that the Federal government should offer incentives to aid the competitive position of secondary materials sector. This paper examines principal economic arguments that have been offered in support of a Federal program of recycling incentives and analyzes some of the recent legislative proposals in light of available information on the structure of the secondary materials industry. Arguments advanced in support of recycling incentives is that tax equity should be established between recyclers and primary material producers. (Depletion deductions were supported in H.R. 148). A second argument is based upon market failure attributable to external diseconomies in primary material production (air and

In [10]:
for record in title_abstract_data:
    record["id"] = record["id"].replace("https://openalex.org/", "")


In [11]:

general_data["id"] = general_data["id"].str.replace("https://openalex.org/", "", regex=False)


In [12]:
ids_set = set(general_data["id"])


In [13]:
# save ids_set efficiently
import pickle
with open("ids_set_top100.pkl", "wb") as file:
    pickle.dump(ids_set, file)

In [47]:
# # save ids_set efficiently
# import pickle
# with open("ids_set_top50.pkl", "wb") as file:
#     pickle.dump(ids_set, file)

In [70]:
# remove general_data from memory
del general_data


In [14]:
# filter titie_abstract_data to include only those with id in general_data
title_abstract_data = [record for record in title_abstract_data if record["id"] in ids_set]

In [16]:

# Deduplicate based on 'id'
unique_title_abstract_data = {record["id"]: record for record in title_abstract_data}.values()

# Convert back to a list
title_abstract_data = list(unique_title_abstract_data)

# rmeove unique_title_abstract_data from memory
del unique_title_abstract_data

# Save the deduplicated data back to a file (optional)
# with open("title_abstract_unique.json", "w", encoding="utf-8") as file:
#     json.dump(title_abstract_data, file, indent=2)

print(f"Number of records after deduplication: {len(title_abstract_data)}")


Number of records after deduplication: 389235


investigate why this happens

# prompts

In [ ]:

import os

In [25]:
import os
import json
import pandas as pd

# Updated response_format with strict schema and required fields
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "research_graph_edges_v1",
        "strict": True,  # Enforce strict schema validation
        "schema": {
            "type": "object",
            "properties": {
                "edges": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "claim": {
                                "type": "string",
                                "description": "The relationship in 'A -> B' format, where A is the source and B is the sink."
                            },
                            "source": {
                                "type": "string",
                                "description": "The source node in the relationship, representing the independent variable, treatment, or cause."
                            },
                            "sink": {
                                "type": "string",
                                "description": "The sink node in the relationship, representing the dependent variable, outcome, or effect."
                            },
                            "is_causal_relationship": {
                                "type": "boolean",
                                "description": "True if the relationship is causal; False otherwise."
                            },
                            "justification": {
                                "type": "string",
                                "description": "A concise explanation of why the relationship is considered causal or not."
                            },
                            "sign_of_impact": {
                                "type": "string",
                                "description": "Direction of the effect.",
                                "enum": ["increase", "decrease", "no effect", "ambiguous", "NA"]
                            },
                            "effect_size": {
                                "type": "string",
                                "description": "Effect size, including magnitude if stated. If not available, write 'NA'."
                            },
                            "statistical_significance": {
                                "type": "string",
                                "description": "Statistical significance of the effect.",
                                "enum": ["p<0.01", "0.01<=p<0.05", "0.05<=p<0.1", "p>0.1", "NA"]
                            },
                            "claim_type": {
                                "type": "string",
                                "description": "Type of claim regarding the relationship.",
                                "enum": [
                                    "effect exists",
                                    "effect does not exist",
                                    "statistically negligible",
                                    "meaningfully negligible",
                                    "no positive effect",
                                    "statistically no positive",
                                    "meaningfully no positive",
                                    "no evidence",
                                    "similar values",
                                    "subset effect",
                                    "short-term effect",
                                    "other"
                                ]
                            },
                            "claim_type_description": {
                                "type": "string",
                                "description": "Provide description if 'other' is selected for claim_type. If not applicable, write 'NA'."
                            },
                            "nature_of_evidence": {
                                "type": "string",
                                "description": "The nature of the evidence used in the study.",
                                "enum": ["quantitative", "qualitative", "mixed methods", "theoretical/conceptual", "NA"]
                            },
                            "data_accessibility": {
                                "type": "object",
                                "properties": {
                                    "uses_data": {
                                        "type": "boolean",
                                        "description": "True if the paper uses any data; False otherwise."
                                    },
                                    "unit_of_analysis": {
                                        "type": "array",
                                        "items": {
                                            "type": "string",
                                            "enum": [
                                                "individual", "group", "household", "organization", "firm", "institution",
                                                "sector", "industry", "region", "country", "global", "other"
                                            ]
                                        },
                                        "description": "List of units of analysis used in the study."
                                    },
                                    "unit_of_analysis_details": {
                                        "type": "string",
                                        "description": "Additional details about the unit of analysis, especially if 'other' is selected."
                                    },
                                    "temporal_context": {
                                        "type": "string",
                                        "description": "The time periods covered by the data, such as years or dates. If not mentioned, write 'NA'."
                                    },
                                    "start_year": {
                                        "type": "array",
                                        "items": {"type": "integer"},
                                        "description": "List of starting years of the data in each analysis. If not mentioned, write `[]`."
                                    },
                                    "end_year": {
                                        "type": "array",
                                        "items": {"type": "integer"},
                                        "description": "List of ending years of the data in each analysis. If not mentioned, write `[]`."
                                    },
                                    "geographical_context_countries": {
                                        "type": "array",
                                        "items": {"type": "string"},
                                        "description": "List of countries using ISO 3166-1 alpha-3 codes (e.g., USA, GBR). If not mentioned, write `[]`."
                                    },
                                    "ownership": {
                                        "type": "array",
                                        "items": {
                                            "type": "string",
                                            "enum": [
                                                "private company", "public sector entity", "academic institution",
                                                "non-profit organization", "individual researcher", "community-generated",
                                                "open source", "other"
                                            ]
                                        },
                                        "description": "Who owns the data."
                                    },
                                    "ownership_details": {
                                        "type": "array",
                                        "items": {"type": "string"},
                                        "description": "Detailed information about the data owners, including organization names, if available. If not mentioned, write `[]`."
                                    }
                                },
                                "required": [
                                    "uses_data",
                                    "unit_of_analysis",
                                    "unit_of_analysis_details",
                                    "temporal_context",
                                    "start_year",
                                    "end_year",
                                    "geographical_context_countries",
                                    "ownership",
                                    "ownership_details"
                                ],
                                "additionalProperties": False  # Disallow extra fields
                            },
                            # Newly Added Fields (replacing the removed ones)
                            "evidence_method": {
                                "type": "string",
                                "description": "The method used to establish the relationship, as explicitly mentioned by the authors.",
                                "enum": [
                                    "RDD", "DID", "RCT", "IV", "Structural", "TWFE",
                                    "Event Study", "Simulations", "Theoretical/Non-Empirical",
                                    "Other", "Do not know"
                                ]
                            },
                            "evidence_method_other_description": {
                                "type": "string",
                                "description": "Provide a description if 'Other' is used for 'evidence_method'. If not applicable, write 'NA'."
                            },
                            "sources_of_exogenous_variation": {
                                "type": "string",
                                "description": "Sources of exogenous variation used for identification, if any. If not mentioned, write 'NA'."
                            },
                            "level_of_tentativeness": {
                                "type": "string",
                                "description": "Level of tentativeness or certainty in the evidence.",
                                "enum": ["certain", "tentative"]
                            }
                        },
                        "required": [
                            "claim",
                            "source",
                            "sink",
                            "is_causal_relationship",
                            "justification",
                            "sign_of_impact",
                            "effect_size",
                            "statistical_significance",
                            "claim_type",
                            "claim_type_description",
                            "nature_of_evidence",
                            "data_accessibility",
                            "evidence_method",
                            "evidence_method_other_description",
                            "sources_of_exogenous_variation",
                            "level_of_tentativeness"
                        ],
                        "additionalProperties": False  # Disallow extra fields in edges
                    }
                }
            },
            "required": ["edges"],  # Ensure 'edges' is required at the root level
            "additionalProperties": False  # Disallow extra fields at the root level
        }
    }
}

# Updated system prompt with clearer instructions and context
system_prompt = (
    "You are an expert assistant analyzing research papers.\n\n"
    "**Task:**\n"
    "- Given a paper's title and abstract, extract all relationships mentioned.\n"
    "- For each relationship, create an entry in an array called **'edges'** with the specified fields.\n\n"
    "**Key Concepts:**\n"
    "- **Node:** Represents an entity or variable mentioned in the text (e.g., a concept, factor, or measurable quantity).\n"
    "- **Edge:** Represents a relationship between two nodes, indicating how one affects or is connected to the other.\n\n"
    "**Instructions:**\n"
    "- **Thoroughness:** Extract all relationships mentioned in title and abstract, including indirect ones.\n"
    "- **Node Representation:** Use exact terms or close approximations from the text for **'source'** and **'sink'** nodes.\n"
    "  - **Source Node:** The independent variable, treatment, or cause.\n"
    "  - **Sink Node:** The dependent variable, outcome, or effect.\n"
    "- **Edge Representation:** In **'claim'**, write the relationship in 'A -> B' format, where A is the source and B is the sink.\n"
    "- **Edge Attributes:** Include all specified fields for each edge.\n"
    "- **Multiple Relationships:** If A affects B and B affects C, include A -> B, B -> C, and A -> C.\n"
    "- **No Interpretation:** Focus only on explicit content; avoid assumptions.\n"
    "- **Unavailable Information:** If a field's information is unavailable, write `'NA'`.\n\n"
    "**Fields for Each Edge:**\n"
    "- **claim** *(string)*: The relationship in 'A -> B' format.\n"
    "- **source** *(string)*: The source node (independent variable, treatment, or cause).\n"
    "- **sink** *(string)*: The sink node (dependent variable, outcome, or effect).\n"
    "- **is_causal_relationship** *(boolean)*: `true` if the relationship is causal; `false` otherwise.\n"
    "- **justification** *(string)*: A concise explanation based on the text.\n"
    "- **sign_of_impact** *(string)*: Direction of the effect. Choose from:\n"
    "  - `'increase'`, `'decrease'`, `'no effect'`, `'ambiguous'`, `'NA'`.\n"
    "- **effect_size** *(string)*: Effect size; if unavailable, write `'NA'`.\n"
    "- **statistical_significance** *(string)*: Choose from:\n"
    "  - `'p<0.01'`, `'0.01<=p<0.05'`, `'0.05<=p<0.1'`, `'p>0.1'`, `'NA'`.\n"
    "- **claim_type** *(string)*: Choose from:\n"
    "  - `'effect exists'`, `'effect does not exist'`, `'statistically negligible'`,\n"
    "    `'meaningfully negligible'`, `'no positive effect'`, `'statistically no positive'`,\n"
    "    `'meaningfully no positive'`, `'no evidence'`, `'similar values'`,\n"
    "    `'subset effect'`, `'short-term effect'`, `'other'`.\n"
    "- **claim_type_description** *(string)*: If `'other'` is selected; else `'NA'`.\n"
    "- **nature_of_evidence** *(string)*: Choose from:\n"
    "  - `'quantitative'`, `'qualitative'`, `'mixed methods'`, `'theoretical/conceptual'`, `'NA'`.\n"
    "- **data_accessibility** *(object)*: Include:\n"
    "  - **uses_data** *(boolean)*: `true` if the paper uses any data; `false` otherwise.\n"
    "  - **unit_of_analysis** *(array of strings)*: e.g., `'individual'`, `'organization'`, `'region'`, etc.\n"
    "  - **unit_of_analysis_details** *(string)*: If `'other'` is selected, provide details; else `'NA'`.\n"
    "  - **temporal_context** *(string)*: Time periods covered; if not mentioned, write `'NA'`.\n"
    "  - **start_year** *(array of integers)*: If not mentioned, write `[]`.\n"
    "  - **end_year** *(array of integers)*: If not mentioned, write `[]`.\n"
    "  - **geographical_context_countries** *(array of strings)*: If not mentioned, write `[]`.\n"
    "  - **ownership** *(array of strings)*: e.g., `'private company'`, `'public sector entity'`, etc.\n"
    "  - **ownership_details** *(array of strings)*: If not mentioned, write `[]`.\n\n"
    "- **evidence_method** *(string)*: The method used to establish the relationship, if stated. Choose from:\n"
    "  - `'RDD'`, `'DID'`, `'RCT'`, `'IV'`, `'Structural'`, `'TWFE'`,\n"
    "    `'Event Study'`, `'Simulations'`, `'Theoretical/Non-Empirical'`,\n"
    "    `'Other'`, `'Do not know'`.\n"
    "- **evidence_method_other_description** *(string)*: If `'Other'` is selected, provide details; else `'NA'`.\n"
    "- **sources_of_exogenous_variation** *(string)*: If any exogenous variation is used; otherwise `'NA'`.\n"
    "- **level_of_tentativeness** *(string)*: `'certain'` or `'tentative'`.\n\n"
    "**Notes:**\n"
    "- Include all required fields; do not add extra fields.\n"
    "- Maintain consistent formatting.\n"
    "- Focus exclusively on content from the texts.\n"
    "- Be thorough in extracting relationships.\n\n"
    "Use the provided texts to extract the relationships as specified."
)

def create_jsonl_entry_from_list(record):
    paper_id = record['id']
    title = record['title']
    abstract = record['abstract']
    
    entry = {
        "custom_id": paper_id,
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4o-mini-2024-07-18",
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": (
                        f"Here is the title of the paper:\n\n{title}\n\n"
                        f"Here is the abstract of the paper:\n\n{abstract}\n\n"
                        "Use these texts to extract the relationships as specified."
                    )
                }
            ],
            "temperature": 0.7,
            "max_tokens": 16000,
            "response_format": response_format
        }
    }
    
    return entry


In [26]:

# Create the JSONL file for batch processing
jsonl_filename = "batch_input_edges_titles_abstracts_top100.jsonl"
with open(jsonl_filename, 'w', encoding='utf-8') as jsonl_file:
    for record in title_abstract_data:  # Directly iterate over the list
        entry = create_jsonl_entry_from_list(record)
        jsonl_file.write(json.dumps(entry) + '\n')

print("JSONL file created successfully.")


JSONL file created successfully.


In [5]:
import os
import json
import random
import time
import pandas as pd
import logging
import pickle
from tqdm import tqdm

# Import the OpenAI client (assuming it's defined similarly to the user's code)
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()


In [28]:
# Function to create a sample JSONL file with 1 random line
def create_sample_jsonl(file_path, sample_size=1):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    # Randomly select sample_size lines from the full file
    sample_lines = random.sample(lines, sample_size)
    
    # Create a smaller JSONL file with just the sample lines
    sample_file_path = "sample_input_titles_abstracts.jsonl"
    with open(sample_file_path, 'w', encoding='utf-8') as sample_file:
        sample_file.writelines(sample_lines)
    
    print(f"Sample of {sample_size} lines created successfully in {sample_file_path}.")
    return sample_file_path

# Function to upload the sample file and create a batch
def upload_and_create_sample_batch(sample_file_path):
    # Upload the sample JSONL file
    sample_input_file = client.files.create(
        file=open(sample_file_path, "rb"),
        purpose="batch"
    )
    
    # Corrected to use dot notation
    sample_input_file_id = sample_input_file.id
    
    # Create a batch with the sample file
    batch = client.batches.create(
        input_file_id=sample_input_file_id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
        metadata={
            "description": "Pilot research paper analysis with sample lines"
        }
    )
    
    print(f"Sample batch job created successfully with Batch ID: {batch.id}")
    return batch.id

# Run the process
sample_file = create_sample_jsonl("batch_input_edges_titles_abstracts_top100.jsonl", 1)
sample_batch_id = upload_and_create_sample_batch(sample_file)


Sample of 1 lines created successfully in sample_input_titles_abstracts.jsonl.
Sample batch job created successfully with Batch ID: batch_678e07a6d6608190a24cc2279b3c023e


In [29]:

# Batches to be in <100mb chunks. Let's split them
# Create input_batches_titles_abstracts_top100 directory if it doesn't exist
if not os.path.exists("input_batches_titles_abstracts_top100"):
    os.makedirs("input_batches_titles_abstracts_top100")

# Function to split JSONL file into smaller chunks
def split_jsonl_file(file_path, num_batches):
    with open(file_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    total_lines = len(lines)
    lines_per_batch = total_lines // num_batches
    
    for i in range(num_batches):
        batch_lines = lines[i * lines_per_batch : (i + 1) * lines_per_batch]
        batch_file_path = f"input_batches_titles_abstracts_top100/batch_input_edges_part{i + 1}.jsonl"
        with open(batch_file_path, 'w', encoding='utf-8') as batch_file:
            batch_file.writelines(batch_lines)
    
    # If there are leftover lines, add them to the last batch
    if total_lines % num_batches != 0:
        with open(f"input_batches_titles_abstracts_top100/batch_input_edges_part{num_batches}.jsonl", 'a', encoding='utf-8') as batch_file:
            batch_file.writelines(lines[num_batches * lines_per_batch :])
    
    print("Batches created successfully.")

# Run the function
split_jsonl_file(jsonl_filename, 50)



Batches created successfully.


In [31]:

# Step 2: Upload Each Batch for Processing

# Function to upload and create batches
def upload_and_create_batches(num_batches):
    batch_ids = []
    for i in range(1, num_batches + 1):
        batch_file_path = f"input_batches_titles_abstracts_top100/batch_input_edges_part{i}.jsonl"
        
        # Upload the JSONL file for batch processing
        batch_input_file = client.files.create(
            file=open(batch_file_path, "rb"),
            purpose="batch"
        )
        
        # Use dot notation to access 'id'
        batch_input_file_id = batch_input_file.id
        
        # Create the batch
        batch = client.batches.create(
            input_file_id=batch_input_file_id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "description": f"Research paper analysis job edges part {i}"
            }
        )
        
        # Use dot notation to access 'id'
        batch_ids.append(batch.id)
        print(f"Batch {i} job created successfully with Batch ID: {batch.id}")
        
        # Pause between uploads to avoid hitting rate limits
        time.sleep(1)
    
    return batch_ids


# Run the function
num_batches = 50  # Adjust the number of batches as needed
batch_ids = upload_and_create_batches(num_batches)


Batch 1 job created successfully with Batch ID: batch_678e0ac5022c819093a46fae29b6efc0
Batch 2 job created successfully with Batch ID: batch_678e0af78140819093a5248150eff580
Batch 3 job created successfully with Batch ID: batch_678e0b3324608190904875837f1e77ee
Batch 4 job created successfully with Batch ID: batch_678e0b600cd88190892f706bbfea6c8c
Batch 5 job created successfully with Batch ID: batch_678e0b8dd85c81909c22c16f9a8c6d1e
Batch 6 job created successfully with Batch ID: batch_678e0bbb67ac8190beee232b9965aa83
Batch 7 job created successfully with Batch ID: batch_678e0be74ccc8190b00e46ff36986858
Batch 8 job created successfully with Batch ID: batch_678e0c13f4f88190b4ae0e8d9fa27570
Batch 9 job created successfully with Batch ID: batch_678e0c4092ac8190a0e8f4365460f359
Batch 10 job created successfully with Batch ID: batch_678e0c6ddfac8190bfb3cd98cdf29e6c
Batch 11 job created successfully with Batch ID: batch_678e0c9a46bc8190867c5f6fdabe1f7a
Batch 12 job created successfully with Ba

In [32]:

# Save batch_ids vector in case we need to reference it later
with open('batch_ids_titles_abstracts_top100.pkl', 'wb') as f:
    pickle.dump(batch_ids, f)


In [6]:
# read batch_ids
with open('batch_ids_titles_abstracts_top100.pkl', 'rb') as f:
    batch_ids = pickle.load(f)

In [7]:

# Step 3: Check the Status of Each Batch and Retrieve the Results

# Create output_batches_titles_abstracts_top100 directory if it doesn't exist
if not os.path.exists("output_batches_titles_abstracts_top100"):
    os.makedirs("output_batches_titles_abstracts_top100")

# Function to check the status of the batches
def check_batch_status(batch_ids):
    for batch_id in batch_ids:
        batch_status = client.batches.retrieve(batch_id)
        print(f"Status for Batch ID {batch_id}: {batch_status.status}")
        
        if batch_status.status == 'completed':
            # Retrieve the output file
            output_file_id = batch_status.output_file_id
            file_response = client.files.content(output_file_id)
            output_file_path = f"output_batches_titles_abstracts_top100/batch_output_edges_{batch_id}.jsonl"
            with open(output_file_path, "w", encoding='utf-8') as output_file:
                output_file.write(file_response.text)
            print(f"Batch output saved to {output_file_path}")
        else:
            print(f"Batch {batch_id} not completed yet. Please check again later.")


# Example usage: Check the status of the batches
check_batch_status(batch_ids)


Status for Batch ID batch_678e0ac5022c819093a46fae29b6efc0: completed
Batch output saved to output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0ac5022c819093a46fae29b6efc0.jsonl
Status for Batch ID batch_678e0af78140819093a5248150eff580: completed
Batch output saved to output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0af78140819093a5248150eff580.jsonl
Status for Batch ID batch_678e0b3324608190904875837f1e77ee: completed
Batch output saved to output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0b3324608190904875837f1e77ee.jsonl
Status for Batch ID batch_678e0b600cd88190892f706bbfea6c8c: completed
Batch output saved to output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0b600cd88190892f706bbfea6c8c.jsonl
Status for Batch ID batch_678e0b8dd85c81909c22c16f9a8c6d1e: completed
Batch output saved to output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0b8dd85c81909c22c16f9a8c6d1e.jsonl
Status for Batch ID batch

In [8]:

# Step 4: Combine the Results from All Batches into a Single DataFrame and Save as CSV

# Function to load a JSONL file into a list of dictionaries
def load_jsonl(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return [json.loads(line) for line in file.readlines()]

# Function to process and normalize the new response format
def normalize_json_edges(data):
    flat_data = []
    for entry in data:
        if (
            'response' in entry
            and 'body' in entry['response']
            and 'choices' in entry['response']['body']
        ):
            paper_id = entry.get('custom_id', 'unknown_id')
            for choice in entry['response']['body']['choices']:
                message_content = choice['message']['content']
                try:
                    # Parse the JSON content from the message
                    content_json = json.loads(message_content)
                    
                    # content_json should be a dictionary with key 'edges'
                    if 'edges' in content_json:
                        edges_list = content_json['edges']
                        for idx, edge in enumerate(edges_list):
                            if isinstance(edge, dict):
                                edge_entry = {}
                                edge_entry['paper_id'] = paper_id
                                edge_entry['edge_number'] = idx + 1  # starting from 1
                                edge_entry['paper_edge_id'] = f"{paper_id}_{idx + 1}"
                                
                                # Extract fields from edge
                                for field in [
                                    # 'claim', 'source', 'sink', 'is_causal_relationship', 'justification',
                                    # 'sign_of_impact', 'effect_size', 'statistical_significance',
                                    # 'claim_type', 'claim_type_description', 'methodological_approach',
                                    # 'methodological_approach_description', 'nature_of_evidence',
                                    # 'specific_methodology', 'data_accessibility'

                                    'claim', 'source', 'sink', 'is_causal_relationship', 'justification',
                                    'sign_of_impact', 'effect_size', 'statistical_significance',
                                    'claim_type', 'claim_type_description', 'nature_of_evidence',
                                    'data_accessibility', 'evidence_method', 
                                    'evidence_method_other_description', 'sources_of_exogenous_variation',
                                    'level_of_tentativeness'
                                ]:
                                    edge_entry[field] = edge.get(field, 'NA')
                                
                                flat_data.append(edge_entry)
                            else:
                                logging.error(f"Edge is not a dictionary for custom_id {paper_id}. Edge: {edge}")
                    else:
                        logging.error(f"No 'edges' key found in content for custom_id {paper_id}. Content: {content_json}")
                except (json.JSONDecodeError, TypeError) as e:
                    logging.error(f"Failed to parse content for custom_id {paper_id}. Error: {e}")
                    logging.error(f"Content: {message_content}")
        else:
            logging.error(f"No valid response found in entry with custom_id {entry.get('custom_id', 'unknown')}")
    
    return pd.DataFrame(flat_data)

# Function to load all batch files and create a DataFrame
def load_and_parse_batches(batch_files_dir):
    batch_files = [
        os.path.join(batch_files_dir, f)
        for f in os.listdir(batch_files_dir)
        if f.endswith('.jsonl')
    ]
    all_data = []

    # Configure logging
    logging.basicConfig(filename='parsing_errors_titles_abstracts_top100.log', level=logging.ERROR,
                        format='%(asctime)s:%(levelname)s:%(message)s')

    # Load and process each batch file with progress tracking
    for batch_file in tqdm(batch_files, desc="Processing batch files"):
        print(f"Processing batch file: {batch_file}")
        batch_data = load_jsonl(batch_file)
        normalized_data = normalize_json_edges(batch_data)
        if not normalized_data.empty:
            all_data.append(normalized_data)

    # Concatenate all DataFrames into one
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data found in batch files.")
        return pd.DataFrame()

# Load and parse the batches into a DataFrame
batch_files_dir = 'output_batches_titles_abstracts_top100/'  # Adjust this to the directory where your batches are located
df_edges = load_and_parse_batches(batch_files_dir)

# Handle missing or improperly formatted entries by replacing NaNs with 'NA'
df_edges.fillna('NA', inplace=True)

# Convert boolean fields from True/False to 'True'/'False' strings if desired
boolean_fields = ['is_causal_relationship', 'uses_data']
for field in boolean_fields:
    if field in df_edges.columns:
        df_edges[field] = df_edges[field].apply(lambda x: 'True' if x == True else ('False' if x == False else x))

# Since 'data_accessibility' is a nested object, we need to flatten it
# Expand 'data_accessibility' into separate columns
if 'data_accessibility' in df_edges.columns:
    data_accessibility_df = df_edges['data_accessibility'].apply(pd.Series)
    # Merge back into df_edges
    df_edges = pd.concat([df_edges.drop(columns=['data_accessibility']), data_accessibility_df], axis=1)

# Save the flattened DataFrame to a CSV
output_csv = 'extracted_edges_titles_abstracts_top100.csv'
df_edges.to_csv(output_csv, index=False)

print(f"All JSON columns have been successfully flattened and saved to {output_csv}.")


Processing batch files:   0%|          | 0/50 [00:00<?, ?it/s]

Processing batch file: output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0ac5022c819093a46fae29b6efc0.jsonl


Processing batch files:   2%|▏         | 1/50 [00:01<01:07,  1.38s/it]

Processing batch file: output_batches_titles_abstracts_top100/batch_output_edges_batch_678e0af78140819093a5248150eff580.jsonl


Processing batch files:   4%|▍         | 2/50 [00:02<01:02,  1.30s/it]

...

Processing batch file: output_batches_titles_abstracts_top100/batch_output_edges_batch_678e13e606dc81908c5c341fe07188a4.jsonl


Processing batch files:  98%|█████████▊| 49/50 [00:52<00:00,  1.03it/s]

Processing batch file: output_batches_titles_abstracts_top100/batch_output_edges_batch_678e14133b44819098a691e434e962ef.jsonl


Processing batch files: 100%|██████████| 50/50 [00:54<00:00,  1.09s/it]
C:\Users\pgarg1\AppData\Local\Temp\ipykernel_22012\2993619358.py:109: FutureWarning: Returning a DataFrame from Series.apply when the supplied function returns a Series is deprecated and will be removed in a future version.
  data_accessibility_df = df_edges['data_accessibility'].apply(pd.Series)


All JSON columns have been successfully flattened and saved to extracted_edges_titles_abstracts_top100.csv.
